<a href="https://colab.research.google.com/github/deruke/aisecops-labs/blob/main/notebooks/Optional-LLM_Phishing_Evasion.ipynb" target="_new"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optional Lab: Can LLM-Generated Phishing Evade AI Detection?
## Using GPT-4 to Craft Adversarial Emails Against a BERT-Based Classifier

**Workshop:** Attacking, Defending & Leveraging Agentic AI
**Authors:** Derek Banks and Dr. Brian Fehrman
**Time:** 45 minutes
**Platform:** Google Colab (CPU) or Docker lab environment
**Theme:** Attack / Defend

---

### What You Will Learn

1. **Load a pre-trained BERT phishing detector** from Hugging Face and establish a detection baseline
2. **Use the OpenAI API** to generate synthetic phishing emails with varied social engineering tactics
3. **Test LLM-generated phishing against the BERT classifier** and measure evasion rates
4. **Analyze the adversarial arms race** between generative AI (offense) and ML detection (defense)
5. **Experiment with evasion techniques** — prompt engineering that bypasses content-based classifiers

### Prerequisites

- Familiarity with Lab 1 (ML-based URL/phishing classification)
- A valid OpenAI API key (set via the Docker environment or manually)
- Basic understanding of how text classifiers work

### Threat Model Connection

This lab demonstrates **MITRE ATLAS ML Attack Lifecycle** techniques:
- **AML.T0043 — Craft Adversarial Data**: Using LLMs to generate inputs that evade ML classifiers
- **AML.T0015 — Evade ML Model**: Testing whether AI-generated content bypasses trained detectors

The core question: *If defenders deploy ML-based phishing detection, can attackers use generative AI to systematically evade it?*

### Lab Roadmap

| Section | What You'll Do |
|---------|---------------|
| **1. Setup** | Verify packages, configure OpenAI API key |
| **2. Load BERT Detector** | Load `ealvaradob/bert-finetuned-phishing` from Hugging Face |
| **3. Baseline Testing** | Test BERT against real phishing emails from a labeled dataset |
| **4. LLM Phishing Generation** | Use GPT-4.1-mini to generate phishing emails with 5 social engineering tactics |
| **5. Evasion Testing** | Run generated emails through BERT and measure detection vs. evasion |
| **6. Advanced Evasion** | Craft prompts specifically designed to evade content-based classifiers |
| **7. Analysis & Visualization** | Compare results, visualize evasion rates, analyze confidence scores |
| **8. Discussion** | Implications for the AI security arms race |

---
## Section 1: Environment Setup

We need `transformers` and `torch` for the BERT phishing detector, `openai` for generating phishing emails, and standard data science libraries for analysis. All of these are pre-installed in the Docker lab environment.

In [ ]:
# ============================================================
# Cell 1: Environment Setup
# ============================================================
# Verify that all required packages are available.
# These are pre-installed in the Docker lab environment.
# If running in Colab, missing packages will be installed.
# ============================================================

import importlib

packages = ['transformers', 'torch', 'openai', 'pandas', 'matplotlib']
missing = []

for pkg in packages:
    try:
        importlib.import_module(pkg)
        print(f'[+] {pkg} is available')
    except ImportError:
        missing.append(pkg)
        print(f'[!] {pkg} is missing')

if missing:
    print(f'\n[!] Installing missing packages: {", ".join(missing)}')
    import subprocess
    subprocess.check_call(['pip', 'install', '-q'] + missing)
    print('[+] Installation complete!')
else:
    print('\n[+] All required packages are available!')

In [ ]:
# ============================================================
# Cell 2: Imports and Configuration
# ============================================================
# Import all libraries and set reproducibility seed.
# ============================================================

import os
import sys
import warnings
import pandas as pd
import torch
import transformers
import openai
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

RANDOM_SEED = 42

print(f'Python version: {sys.version}')
print(f'PyTorch version: {torch.__version__}')
print(f'Transformers version: {transformers.__version__}')

In [ ]:
# ============================================================
# Cell 3: OpenAI API Key Configuration
# ============================================================
# The OPENAI_API_KEY is passed into the Docker container via
# the .env file. If running in Colab, set it manually below.
# ============================================================

api_key = os.environ.get("OPENAI_API_KEY", "")
if not api_key:
    print("WARNING: OPENAI_API_KEY is not set!")
    print("Set it in your .env file and restart the container, or set it here:")
    print('  os.environ["OPENAI_API_KEY"] = "sk-..."')
else:
    print(f"OPENAI_API_KEY is set (ends with ...{api_key[-4:]})")

# Uncomment and fill in if running outside Docker:
# os.environ["OPENAI_API_KEY"] = "sk-..."

---
## Section 2: Load the BERT Phishing Detector

We'll use [`ealvaradob/bert-finetuned-phishing`](https://huggingface.co/ealvaradob/bert-finetuned-phishing) from Hugging Face — a BERT model fine-tuned specifically for phishing email detection.

**Model Details:**
- **Base:** Google's BERT (Bidirectional Encoder Representations from Transformers)
- **Task:** Binary text classification — `phishing` or `benign`
- **Claimed Performance:** 97.17% accuracy, 96.58% precision, 96.70% recall

This is our **defender** — the ML model that an attacker needs to evade.

In [ ]:
# ============================================================
# Cell 4: Load the BERT Phishing Detection Model
# ============================================================
# Downloads the pre-trained model from Hugging Face and creates
# a classification pipeline. The model is cached after first
# download, so subsequent loads are fast.
# ============================================================

# Detect available hardware
device = 'cpu'
if torch.cuda.is_available():
    device = 'cuda'
    print('[+] NVIDIA GPU detected - using CUDA')
elif torch.backends.mps.is_available():
    device = 'mps'
    print('[+] Apple Silicon GPU detected - using MPS')
else:
    print('[*] No GPU detected - using CPU (this is fine for inference)')

# Load the pre-trained phishing detection model
model_name = "ealvaradob/bert-finetuned-phishing"
print(f'\n[*] Loading model: {model_name}')

tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

# Create a prediction pipeline
phishing_detector = transformers.pipeline(
    'text-classification',
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True
)

print('[+] Phishing detection model loaded successfully!')

In [ ]:
# ============================================================
# Cell 5: Quick Sanity Check
# ============================================================
# Verify the model works with obvious examples before we
# start generating adversarial content.
# ============================================================

sanity_tests = [
    ("Benign", "Hi Sarah, just confirming our 2pm meeting tomorrow. I'll bring the Q3 report."),
    ("Phishing", "URGENT: Your account has been compromised! Click here immediately to verify: http://secure-bank-login.com/verify"),
    ("Benign", "Thanks for the great presentation today! I'll send the slides by end of day."),
    ("Phishing", "Dear customer, we noticed suspicious activity on your PayPal account. Confirm your details within 24 hours or your account will be suspended."),
]

print("Sanity Check — Testing with obvious examples:\n")
print(f"{'Type':<10} {'Predicted':<12} {'Confidence':<12} {'Text Preview'}")
print("=" * 80)

for expected, text in sanity_tests:
    result = phishing_detector(text)[0]
    label = result['label']
    score = result['score']
    preview = text[:50] + "..." if len(text) > 50 else text
    match = "OK" if (label == "phishing") == (expected == "Phishing") else "MISMATCH"
    print(f"{expected:<10} {label:<12} {score:<12.1%} {preview} [{match}]")

---
## Section 3: Baseline — Testing Against Real Phishing Emails

Before we generate adversarial content, let's establish a baseline. We'll test the BERT model against real phishing and legitimate emails from the [Kaggle Phishing Email Dataset](https://www.kaggle.com/datasets/subhajournal/phishing-emails) — the same dataset used in training phishing detection models.

This tells us: **How good is the defender under normal conditions?**

In [ ]:
# ============================================================
# Cell 6: Load and Sample the Phishing Email Dataset
# ============================================================
# Load a labeled phishing dataset, clean it, and select a
# balanced random sample for baseline evaluation.
# ============================================================

# Check for local copy first (pre-downloaded in Docker), otherwise download
local_path = '/home/jovyan/datasets/Phishing_Email.csv.gz'
remote_url = 'https://raw.githubusercontent.com/RiverGumSecurity/Datasets/refs/heads/main/Kaggle/Phishing_Email.csv.gz'

print('[*] Loading phishing email dataset...')

if os.path.exists(local_path):
    print(f'[+] Using pre-downloaded dataset from {local_path}')
    df_emails = pd.read_csv(local_path)
else:
    print(f'[*] Downloading dataset from remote URL...')
    df_emails = pd.read_csv(remote_url)

# Clean up
df_emails = df_emails.drop(['Unnamed: 0'], axis=1, errors='ignore')
df_emails = df_emails.dropna()
df_emails = df_emails.drop_duplicates()

print(f'[+] Dataset loaded: {len(df_emails):,} emails')
print(f'\nEmail Type Distribution:')
print(df_emails['Email Type'].value_counts())

In [ ]:
# ============================================================
# Cell 7: Baseline Evaluation — Real Emails vs. BERT
# ============================================================
# Test 10 random emails (5 phishing, 5 safe) against the
# BERT classifier to establish baseline accuracy.
# ============================================================

# Balanced sample
phishing_samples = df_emails[df_emails['Email Type'] == 'Phishing Email'].sample(n=5, random_state=RANDOM_SEED)
safe_samples = df_emails[df_emails['Email Type'] == 'Safe Email'].sample(n=5, random_state=RANDOM_SEED)
baseline_samples = pd.concat([phishing_samples, safe_samples]).sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)

# Run predictions
baseline_results = []
correct = 0

print("BASELINE: BERT vs. Real Emails from Dataset\n")
print(f"{'#':<4} {'Actual':<12} {'Predicted':<12} {'Confidence':<12} {'Correct'}")
print("=" * 55)

for idx, row in baseline_samples.iterrows():
    email_text = row['Email Text']
    actual_label = row['Email Type']
    actual_simple = 'phishing' if actual_label == 'Phishing Email' else 'benign'

    prediction = phishing_detector(email_text)[0]
    predicted_label = prediction['label']
    confidence = prediction['score']

    is_correct = (predicted_label == actual_simple)
    if is_correct:
        correct += 1

    baseline_results.append({
        'actual': actual_simple,
        'predicted': predicted_label,
        'confidence': confidence,
        'correct': is_correct,
        'text': email_text[:200]
    })

    mark = "Y" if is_correct else "N"
    print(f"{len(baseline_results):<4} {actual_simple:<12} {predicted_label:<12} {confidence:<12.1%} {mark}")

accuracy = correct / len(baseline_samples) * 100
print(f"\n{'=' * 55}")
print(f"BASELINE ACCURACY: {correct}/{len(baseline_samples)} ({accuracy:.0f}%)")
print(f"{'=' * 55}")

---
## Section 4: The Attacker — Generating Phishing with GPT-4.1-mini

Now we switch to the **attacker's perspective**. We'll use OpenAI's GPT-4.1-mini to generate phishing emails using 5 common social engineering tactics. Each tactic represents a real-world phishing strategy:

| # | Tactic | Real-World Example |
|---|--------|-------------------|
| 1 | **Urgency + Account Compromise** | Fake bank fraud alerts |
| 2 | **IT Support / Password Reset** | "Your password expires today" |
| 3 | **Package Delivery** | FedEx/UPS tracking scams |
| 4 | **Business Email Compromise (BEC)** | Fake invoice from "CEO" |
| 5 | **Social Media Security** | "Your Instagram was accessed from a new device" |

**Ethical Note:** This lab generates phishing content strictly for security research and awareness training. Understanding attacker techniques is essential for building effective defenses.

In [ ]:
# ============================================================
# Cell 8: OpenAI Generation Function
# ============================================================
# Helper function to generate text via the OpenAI API.
# Uses the Chat Completions endpoint with gpt-4.1-mini.
# ============================================================

def generate_email(prompt, system_prompt=None, model="gpt-4.1-mini"):
    """
    Generate text using OpenAI's Chat Completions API.

    Args:
        prompt: The user instruction describing what to generate
        system_prompt: Optional system-level context
        model: OpenAI model ID

    Returns:
        Generated text string
    """
    client = openai.OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))

    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        max_tokens=1024,
        temperature=0.7
    )

    return response.choices[0].message.content

print('[+] generate_email() function defined')

In [ ]:
# ============================================================
# Cell 9: Define Phishing Generation Prompts (5 Tactics)
# ============================================================
# Each prompt instructs GPT-4.1-mini to generate a realistic
# phishing email using a specific social engineering tactic.
# The system prompt frames this as security research.
# ============================================================

SYSTEM_PROMPT = """You are a cybersecurity researcher creating phishing email examples
for security awareness training. Generate realistic but clearly educational examples
that demonstrate common phishing tactics. Output ONLY the email body — no commentary,
no subject line label, no disclaimers. The email should look like it came from a
real sender."""

phishing_prompts = {
    "Urgency / Account Compromise": (
        "Write a phishing email that pretends to be from a major bank warning about "
        "suspicious account activity. Use urgency, fear of financial loss, and include "
        "a fake verification link."
    ),
    "IT Support / Password Reset": (
        "Write a phishing email that impersonates the company IT department, claiming "
        "the recipient's password expires today and they must update it immediately "
        "through a provided link."
    ),
    "Package Delivery": (
        "Write a phishing email that pretends to be from a shipping company about a "
        "package that couldn't be delivered. Include a fake tracking link and request "
        "the recipient confirm their delivery address."
    ),
    "Business Email Compromise (BEC)": (
        "Write a phishing email that appears to be from the company CFO requesting "
        "urgent payment of an invoice to a new vendor. Use professional business "
        "language and create time pressure."
    ),
    "Social Media Security Alert": (
        "Write a phishing email that impersonates a social media platform warning "
        "about a login from an unrecognized device. Include a fake security "
        "verification link."
    ),
}

print(f'[+] Defined {len(phishing_prompts)} phishing generation prompts')
for tactic in phishing_prompts:
    print(f'    - {tactic}')

---
## Section 5: Generate and Test — The Evasion Experiment

Now we run the core experiment:

1. **Generate** a phishing email for each tactic using GPT-4.1-mini
2. **Classify** each generated email with the BERT phishing detector
3. **Record** whether the email was detected or evaded detection

This simulates what a real attacker would do: generate phishing content with AI and test it against known defenses before deploying it.

In [ ]:
# ============================================================
# Cell 10: Generate Phishing Emails and Test Against BERT
# ============================================================
# For each social engineering tactic, generate a phishing email
# with GPT-4.1-mini, then classify it with the BERT detector.
# This is the core of the adversarial experiment.
# ============================================================

standard_results = []

print("EXPERIMENT: LLM-Generated Phishing vs. BERT Detector\n")
print("=" * 70)

for tactic, prompt in phishing_prompts.items():
    print(f"\n--- Tactic: {tactic} ---")

    try:
        # Generate the phishing email
        generated_email = generate_email(prompt, SYSTEM_PROMPT)

        # Classify with BERT
        detection = phishing_detector(generated_email)[0]
        label = detection['label']
        confidence = detection['score']

        # Store result
        standard_results.append({
            'tactic': tactic,
            'email': generated_email,
            'detection': label,
            'confidence': confidence,
            'evaded': label == 'benign'
        })

        # Display
        print(f"\nGenerated Email (first 300 chars):")
        print("-" * 40)
        print(generated_email[:300] + ("..." if len(generated_email) > 300 else ""))
        print("-" * 40)
        print(f"Detection: {label.upper()} ({confidence:.1%} confidence)")

        if label == 'benign':
            print(">> EVADED — The detector classified this as benign!")
        else:
            print(">> DETECTED — The detector caught this as phishing.")

    except Exception as e:
        print(f"ERROR: {e}")
        standard_results.append({
            'tactic': tactic, 'email': f'ERROR: {e}',
            'detection': 'error', 'confidence': 0, 'evaded': False
        })

# Summary
valid = [r for r in standard_results if r['detection'] != 'error']
evaded = sum(1 for r in valid if r['evaded'])
print(f"\n{'=' * 70}")
print(f"STANDARD TACTICS: {evaded}/{len(valid)} evaded detection")
print(f"{'=' * 70}")

---
## Section 6: Advanced Evasion — Prompt Engineering for Classifier Bypass

An attacker who knows the defender uses content-based classification can adapt. Let's test prompts specifically designed to produce phishing emails that **avoid the linguistic patterns** BERT was trained to detect:

| Strategy | Why It Might Work |
|----------|------------------|
| **Conversational tone** | Avoids formal/urgent language BERT associates with phishing |
| **Professional business style** | Mimics legitimate corporate email patterns |
| **Context-aware follow-up** | References a "previous conversation" to appear authentic |

These represent the **next level** of adversarial sophistication — the attacker isn't just generating phishing, they're engineering it to bypass specific defenses.

In [ ]:
# ============================================================
# Cell 11: Advanced Evasion Prompts
# ============================================================
# These prompts instruct the LLM to generate phishing emails
# that specifically avoid patterns ML classifiers look for:
# urgency keywords, threatening language, suspicious URLs.
# ============================================================

EVASION_SYSTEM_PROMPT = """You are a cybersecurity researcher testing ML-based phishing
detectors. Generate a phishing email that would be difficult for an automated text
classifier to detect. The email should still accomplish its phishing goal (get the
recipient to click a link or provide information) but should avoid:
- Urgent or threatening language
- ALL CAPS words
- Obvious fake URLs
- Phrases commonly associated with phishing (verify, suspend, unauthorized)

Output ONLY the email body. Make it look like a genuine, everyday email."""

evasion_prompts = {
    "Conversational / Friendly Tone": (
        "Write a phishing email disguised as a friendly message from a colleague "
        "sharing a Google Doc link for a project they're working on. Use casual, "
        "natural language like a real coworker would. Include a link the recipient "
        "would want to click."
    ),
    "Professional Business / Routine": (
        "Write a phishing email that looks like a routine quarterly IT compliance "
        "check. Use dry, corporate language. Ask the recipient to log in to a portal "
        "to confirm their access permissions. Make it sound boring and bureaucratic — "
        "the kind of email people click through without thinking."
    ),
    "Context-Aware Follow-Up": (
        "Write a phishing email that appears to be a follow-up to a meeting that "
        "happened yesterday. Reference specific but generic work topics (Q4 planning, "
        "budget review). Include a link to 'meeting notes' that the recipient would "
        "naturally want to open."
    ),
}

print(f'[+] Defined {len(evasion_prompts)} advanced evasion prompts')
for strategy in evasion_prompts:
    print(f'    - {strategy}')

In [ ]:
# ============================================================
# Cell 12: Run Advanced Evasion Experiment
# ============================================================
# Generate evasion-optimized phishing and test against BERT.
# Compare results to the standard tactics from Section 5.
# ============================================================

evasion_results = []

print("ADVANCED EVASION: Classifier-Aware Phishing vs. BERT\n")
print("=" * 70)

for strategy, prompt in evasion_prompts.items():
    print(f"\n--- Strategy: {strategy} ---")

    try:
        generated_email = generate_email(prompt, EVASION_SYSTEM_PROMPT)
        detection = phishing_detector(generated_email)[0]
        label = detection['label']
        confidence = detection['score']

        evasion_results.append({
            'tactic': strategy,
            'email': generated_email,
            'detection': label,
            'confidence': confidence,
            'evaded': label == 'benign'
        })

        print(f"\nGenerated Email (first 300 chars):")
        print("-" * 40)
        print(generated_email[:300] + ("..." if len(generated_email) > 300 else ""))
        print("-" * 40)
        print(f"Detection: {label.upper()} ({confidence:.1%} confidence)")

        if label == 'benign':
            print(">> EVADED — Classifier-aware prompt succeeded!")
        else:
            print(">> DETECTED — BERT still caught it despite evasion attempt.")

    except Exception as e:
        print(f"ERROR: {e}")
        evasion_results.append({
            'tactic': strategy, 'email': f'ERROR: {e}',
            'detection': 'error', 'confidence': 0, 'evaded': False
        })

# Summary
valid_evasion = [r for r in evasion_results if r['detection'] != 'error']
evaded_count = sum(1 for r in valid_evasion if r['evaded'])
print(f"\n{'=' * 70}")
print(f"ADVANCED EVASION: {evaded_count}/{len(valid_evasion)} evaded detection")
print(f"{'=' * 70}")

---
## Section 7: Analysis and Visualization

Let's combine all results and visualize the adversarial arms race. We'll compare:
- **Standard tactics** — straightforward phishing generation
- **Advanced evasion** — classifier-aware prompt engineering

This comparison shows whether an attacker gains an advantage by understanding the defender.

In [ ]:
# ============================================================
# Cell 13: Combined Results Summary
# ============================================================
# Build a combined DataFrame of all experiments and display
# a summary comparison table.
# ============================================================

# Build DataFrames
df_standard = pd.DataFrame([r for r in standard_results if r['detection'] != 'error'])
df_standard['approach'] = 'Standard Tactics'

df_evasion = pd.DataFrame([r for r in evasion_results if r['detection'] != 'error'])
df_evasion['approach'] = 'Advanced Evasion'

df_all = pd.concat([df_standard, df_evasion], ignore_index=True)

# Summary table
print("COMBINED RESULTS SUMMARY\n")
print(f"{'Approach':<22} {'Generated':<12} {'Detected':<12} {'Evaded':<12} {'Evasion Rate'}")
print("=" * 70)

for approach in ['Standard Tactics', 'Advanced Evasion']:
    subset = df_all[df_all['approach'] == approach]
    total = len(subset)
    evaded = subset['evaded'].sum()
    detected = total - evaded
    rate = evaded / total * 100 if total > 0 else 0
    print(f"{approach:<22} {total:<12} {detected:<12} {evaded:<12} {rate:.0f}%")

total_all = len(df_all)
total_evaded = df_all['evaded'].sum()
print(f"\n{'OVERALL':<22} {total_all:<12} {total_all - total_evaded:<12} {total_evaded:<12} {total_evaded / total_all * 100:.0f}%")

In [ ]:
# ============================================================
# Cell 14: Visualization — Detection vs. Evasion
# ============================================================
# Three-panel visualization:
# 1. Per-tactic detection results (bar chart)
# 2. Confidence score comparison
# 3. Overall evasion rate pie chart
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Per-Tactic Results ---
tactics = df_all['tactic']
colors = ['#ef4444' if d == 'phishing' else '#22c55e' for d in df_all['detection']]
bars = axes[0].barh(range(len(tactics)), df_all['confidence'], color=colors)
axes[0].set_yticks(range(len(tactics)))
axes[0].set_yticklabels([t[:25] + "..." if len(t) > 25 else t for t in tactics], fontsize=8)
axes[0].set_xlabel('Detection Confidence')
axes[0].set_title('Per-Tactic Detection Results', fontweight='bold')
axes[0].set_xlim(0, 1)
axes[0].axvline(x=0.5, color='gray', linestyle='--', alpha=0.5)

# Legend
from matplotlib.patches import Patch
axes[0].legend(handles=[
    Patch(facecolor='#ef4444', label='Detected as Phishing'),
    Patch(facecolor='#22c55e', label='Evaded (Classified Benign)')
], loc='lower right', fontsize=7)

# --- Panel 2: Confidence by Approach ---
for i, approach in enumerate(['Standard Tactics', 'Advanced Evasion']):
    subset = df_all[df_all['approach'] == approach]
    x_pos = [i] * len(subset)
    colors_scatter = ['#ef4444' if d == 'phishing' else '#22c55e' for d in subset['detection']]
    axes[1].scatter(x_pos, subset['confidence'], c=colors_scatter, s=100, zorder=5, edgecolors='black')

axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Standard\nTactics', 'Advanced\nEvasion'])
axes[1].set_ylabel('Confidence Score')
axes[1].set_title('Confidence Distribution by Approach', fontweight='bold')
axes[1].set_ylim(0, 1.05)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

# --- Panel 3: Overall Pie Chart ---
detected_count = len(df_all[~df_all['evaded']])
evaded_count_total = len(df_all[df_all['evaded']])
axes[2].pie(
    [detected_count, evaded_count_total],
    labels=['Detected', 'Evaded'],
    autopct='%1.0f%%',
    colors=['#ef4444', '#22c55e'],
    startangle=90,
    textprops={'fontsize': 12}
)
axes[2].set_title('Overall Detection Rate', fontweight='bold')

plt.suptitle('LLM-Generated Phishing vs. BERT Phishing Detector', fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Cell 15: Detailed Evasion Analysis
# ============================================================
# Examine which emails evaded detection and why. Look at
# the specific language patterns that fooled the classifier.
# ============================================================

evaded_emails = df_all[df_all['evaded'] == True]

if len(evaded_emails) > 0:
    print(f"=== EMAILS THAT EVADED DETECTION ({len(evaded_emails)} found) ===\n")

    for _, row in evaded_emails.iterrows():
        print(f"Tactic: {row['tactic']}")
        print(f"Approach: {row['approach']}")
        print(f"Detection: {row['detection']} ({row['confidence']:.1%} confidence)")
        print(f"\nFull Email:")
        print("-" * 50)
        print(row['email'])
        print("-" * 50)
        print()
else:
    print("All LLM-generated phishing emails were detected!")
    print("\nThis is a strong result for the BERT detector.")
    print("However, note that:")
    print("  - We only tested 8 prompts (a real attacker would try thousands)")
    print("  - Temperature and model choice affect output style")
    print("  - More sophisticated evasion techniques exist")

# Show detected emails with lowest confidence (closest to evasion)
detected = df_all[df_all['evaded'] == False].sort_values('confidence')
if len(detected) > 0:
    print(f"\n=== CLOSEST TO EVASION (lowest confidence detections) ===\n")
    for _, row in detected.head(3).iterrows():
        print(f"  {row['tactic']:<30} — {row['detection']} at {row['confidence']:.1%} confidence")
    print(f"\n  These are the phishing emails the detector was least sure about.")

---
## Section 8: Interactive Testing — Try Your Own Prompts

Use the cells below to experiment with your own phishing generation prompts and test them against the BERT detector. Can you craft a prompt that generates phishing content the model misses?

In [ ]:
# ============================================================
# Cell 16: Generate Your Own Phishing Email
# ============================================================
# Modify the prompt below to test your own social engineering
# tactic. Can you craft a prompt that evades the detector?
# ============================================================

# TODO: Modify this prompt to test your own phishing tactic
custom_prompt = """
Write a phishing email that appears to be from HR about updating
direct deposit information for the upcoming payroll cycle.
"""

custom_system = EVASION_SYSTEM_PROMPT  # Or try SYSTEM_PROMPT for standard generation

print("[*] Generating email from your custom prompt...\n")
custom_email = generate_email(custom_prompt, custom_system)

print("Generated Email:")
print("-" * 60)
print(custom_email)
print("-" * 60)

result = phishing_detector(custom_email)[0]
print(f"\nDetection: {result['label'].upper()} ({result['score']:.1%} confidence)")

if result['label'] == 'benign':
    print(">> Your prompt evaded detection!")
else:
    print(">> The detector caught it. Try adjusting your prompt.")

In [ ]:
# ============================================================
# Cell 17: Test Any Email Text Directly
# ============================================================
# Paste any email text below to classify it with the BERT
# detector — useful for testing real emails you receive.
# ============================================================

# TODO: Replace with any email text you want to test
test_email = """
Hi team,

Just a heads up — I've uploaded the updated project timeline to our
shared drive. Could you take a look when you get a chance and let me
know if the deadlines work for your team?

Here's the link: https://docs.google.com/spreadsheets/d/1aBcDeFgHiJk

Thanks!
Alex
"""

result = phishing_detector(test_email)[0]
print(f"Classification: {result['label'].upper()}")
print(f"Confidence: {result['score']:.1%}")

---
## Section 9: Conclusions and Discussion

### Key Findings

From this lab, we observed:

1. **LLMs can generate convincing phishing content at scale** — A single API call produces a realistic email with any social engineering tactic
2. **Standard phishing generation may or may not evade BERT** — Results vary by tactic and generation run due to LLM stochasticity
3. **Classifier-aware prompting changes the game** — When attackers understand how detection works, they can engineer around it
4. **Confidence scores reveal vulnerability** — Even "detected" emails sometimes have low confidence, indicating the classifier is uncertain

### The Adversarial Arms Race

This lab illustrates a fundamental dynamic in AI security:

| Round | Attacker | Defender |
|-------|----------|----------|
| 1 | Sends traditional phishing | Trains BERT on phishing corpus |
| 2 | Uses LLMs to generate novel phishing | BERT catches some, misses others |
| 3 | Engineers prompts to avoid detection patterns | Needs to retrain on LLM-generated samples |
| 4 | Adapts again... | Adapts again... |

**Neither side wins permanently.** This is why defense-in-depth matters: content classifiers, URL reputation, sender verification, and user training all work together.

### Implications for Security Teams

**Offensive Takeaways:**
- LLMs dramatically lower the barrier to generating phishing at scale
- Personalized, context-aware phishing becomes trivially easy
- Attackers can A/B test against detectors before deploying

**Defensive Takeaways:**
- ML-only phishing detection is insufficient against LLM-generated content
- Training data must include LLM-generated examples to stay current
- Multi-signal detection (content + metadata + behavioral) is essential
- User security awareness training remains the last line of defense

### Discussion Questions

1. If an attacker can generate 1,000 phishing variants per minute, how should defenders adapt their detection strategy?
2. Should AI companies restrict the generation of phishing content, even for security research? What are the tradeoffs?
3. What signals beyond email text content could help detect LLM-generated phishing?
4. How does this lab change your perspective on "97% accuracy" as a security metric?

---
## Challenge Exercises

### Exercise 1: Volume Testing

Generate 20+ phishing emails using different temperature settings (0.3, 0.7, 1.0) and calculate the overall evasion rate for each. Does higher temperature (more creative output) help or hurt evasion?

### Exercise 2: Iterative Refinement (Feedback Loop)

Implement an attacker feedback loop: generate a phishing email, test it, and if detected, feed the detection result back to GPT-4 to generate an improved version. How many iterations does it take to evade?

### Exercise 3: Cross-Model Comparison

If you have access to other LLMs (Anthropic Claude, local Ollama models), generate phishing emails with each and compare evasion rates. Do different LLMs produce different detection signatures?

In [ ]:
# ============================================================
# Cell 18: Exercise 2 Starter — Iterative Refinement Loop
# ============================================================
# This implements the attacker feedback loop from Exercise 2.
# The LLM generates phishing, tests it, and if detected,
# asks GPT-4 to improve it based on the detection feedback.
# ============================================================

# TODO: Uncomment and run to try the iterative refinement loop

# MAX_ITERATIONS = 5
# target_tactic = "IT credential harvesting email"
#
# refinement_prompt = target_tactic
# refinement_system = EVASION_SYSTEM_PROMPT
#
# print(f"ITERATIVE REFINEMENT: Attempting to evade detection\n")
# print(f"Starting tactic: {target_tactic}")
# print("=" * 60)
#
# for iteration in range(1, MAX_ITERATIONS + 1):
#     print(f"\n--- Iteration {iteration}/{MAX_ITERATIONS} ---")
#
#     # Generate
#     email = generate_email(refinement_prompt, refinement_system)
#     result = phishing_detector(email)[0]
#     label = result['label']
#     confidence = result['score']
#
#     print(f"Detection: {label} ({confidence:.1%})")
#     print(f"Preview: {email[:150]}...")
#
#     if label == 'benign':
#         print(f"\n>> EVADED on iteration {iteration}!")
#         print(f"\nWinning email:")
#         print(email)
#         break
#
#     # Feed detection result back to the LLM for refinement
#     refinement_prompt = (
#         f"The following phishing email was DETECTED by an ML classifier "
#         f"with {confidence:.0%} confidence:\n\n{email}\n\n"
#         f"Rewrite it to evade detection. Avoid urgency words, suspicious "
#         f"URLs, and phrases commonly associated with phishing. Make it "
#         f"sound like a natural, everyday work email."
#     )
# else:
#     print(f"\n>> Did not evade after {MAX_ITERATIONS} iterations.")
#     print("Try adjusting the refinement prompt strategy.")